# 11 · Experimentos nocturnos — FE v4

Tres experimentos en cadena sobre `dataset_fe_v4.parquet`.  
Cada uno guarda su checkpoint y su submission independientemente.

| Exp | Descripción | Submissions | Tiempo est. |
|-----|-------------|-------------|-------------|
| A | Oversampling manual A/B/C — sin Optuna | `submissions7/` | ~30 min |
| B | Kitchen Sink — 50 trials adicionales desde v5 | `submissions8/` | ~2-3 hs |
| C | Train con 4 meses completos, params v5 | `submissions9/` | ~30 min |

**Orden:** A → B → C en secuencia. Cada uno checkpointea sus probs antes de generar CSVs.  
**Si el runtime muere:** volver a correr desde la celda de setup — cada experimento detecta  
su propio checkpoint y retoma.

**Proyecto:** Predicción de Churn Bancario · Data Mining en EyF  
**Deadline:** 11/10/2026 23:59:59

## 0 · Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
!pip install optuna lightgbm --quiet

In [ ]:
import gc
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import lightgbm as lgb
import optuna
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.rcParams['figure.dpi'] = 110

RUTA_BASE = Path('/content/drive/MyDrive/Maestría Data Mining/DM en EyF')
RUTA_COMP = RUTA_BASE / 'Competencia 1'

# ── Submissions (una por experimento) ─────────────────────────────────────────
RUTA_SUBS_A = RUTA_COMP / 'submissions7'   # oversampling
RUTA_SUBS_B = RUTA_COMP / 'submissions8'   # kitchen sink extended
RUTA_SUBS_C = RUTA_COMP / 'submissions9'   # 4 meses completos
for r in [RUTA_SUBS_A, RUTA_SUBS_B, RUTA_SUBS_C]:
    r.mkdir(parents=True, exist_ok=True)

# ── Checkpoints ───────────────────────────────────────────────────────────────
CKPT_FE_V4       = RUTA_COMP / 'dataset_fe_v4.parquet'
CKPT_V5          = RUTA_COMP / 'mejores_params_optuna_v5.json'
CKPT_V11B        = RUTA_COMP / 'mejores_params_optuna_v11b.json'
CKPT_PROBS_A     = RUTA_COMP / 'probs_v11a.npy'
CKPT_IDS_A       = RUTA_COMP / 'ids_v11a.npy'
CKPT_PROBS_B     = RUTA_COMP / 'probs_v11b.npy'
CKPT_IDS_B       = RUTA_COMP / 'ids_v11b.npy'
CKPT_PROBS_C     = RUTA_COMP / 'probs_v11c.npy'
CKPT_IDS_C       = RUTA_COMP / 'ids_v11c.npy'
ARCHIVO_METRICAS = RUTA_COMP / 'metricas_experimentos.csv'

COL_ID      = 'numero_de_cliente'
COL_PERIODO = 'foto_mes'
TARGET_RAW  = 'clase_ternaria'
TARGET      = 'es_baja2'

MESES_CON_TARGET = [202103, 202104, 202105, 202106]
MES_PREDECIR     = 202108
GANANCIA_TP      =  1_072_500
COSTO_FP         =    -27_500
SEED             = 42
SEMILLAS         = [42, 123, 777, 2024]
N_TRIALS_B       = 50

VARS_PERFIL = ['cliente_antiguedad', 'ctrx_quarter', 'mcuentas_saldo']

def mem_mb(df): return df.memory_usage(deep=True).sum() / 1024**2

print('Checkpoints:')
for ck, nombre in [(CKPT_FE_V4,'FE v4'), (CKPT_V5,'params v5'), (CKPT_V11B,'params v11b')]:
    print(f'  {nombre}: {"✓" if ck.exists() else "✗"}')

## 1 · Funciones base

In [ ]:
def ganancia_por_threshold(y_true, y_prob, n=500):
    thresholds = np.linspace(0.001, 0.999, n)
    rows = []
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        tp   = ((y_true == 1) & (pred == 1)).sum()
        fp   = ((y_true == 0) & (pred == 1)).sum()
        rows.append({'threshold': t,
                     'ganancia':  tp * GANANCIA_TP + fp * COSTO_FP,
                     'tp': int(tp), 'fp': int(fp),
                     'n_intervenidos': int(tp + fp)})
    return pd.DataFrame(rows)

def threshold_optimo(y_true, y_prob):
    df_t = ganancia_por_threshold(y_true, y_prob)
    return df_t.loc[df_t['ganancia'].idxmax()].to_dict()

def evaluar_fold(y_true, y_prob, label=''):
    auc = roc_auc_score(y_true, y_prob)
    opt = threshold_optimo(y_true, y_prob)
    print(f'  [{label}] AUC={auc:.4f} | Ganancia=${opt["ganancia"]:,.0f} | '
          f'N={opt["n_intervenidos"]:,} | TP={opt["tp"]} | thresh={opt["threshold"]:.4f}')
    return {'auc': round(auc, 4), **opt}

def guardar_metricas(rows_nuevas):
    df_n = pd.DataFrame(rows_nuevas)
    df_n['timestamp'] = datetime.now().isoformat()
    if ARCHIVO_METRICAS.exists():
        hist = pd.read_csv(ARCHIVO_METRICAS)
        pd.concat([hist, df_n], ignore_index=True).to_csv(ARCHIVO_METRICAS, index=False)
    else:
        df_n.to_csv(ARCHIVO_METRICAS, index=False)

def generar_submission(probs_list, ids_pred, n_opt, ruta_subs, prefijo, ts):
    paso     = max(100, int(n_opt * 0.025))
    cortes   = [max(8000, min(15000, n_opt + i * paso)) for i in range(-2, 3)]
    archivos = []
    for i_s, semilla in enumerate(SEMILLAS):
        prob_s = probs_list[i_s]
        for n_corte in cortes:
            ranking = np.argsort(prob_s)[::-1][:n_corte]
            ids_int = ids_pred[ranking].astype(int)
            nombre  = f'{prefijo}_sem{semilla}_top{n_corte}_{ts}'
            pd.Series(ids_int).to_csv(ruta_subs / f'{nombre}.csv',
                                      index=False, header=False)
            archivos.append(nombre)
    pd.DataFrame({'nombre': archivos}).to_csv(ruta_subs / f'log_{ts}.csv', index=False)
    # Verificar notación científica
    df_v    = pd.read_csv(ruta_subs / f'{archivos[0]}.csv', header=None)
    tiene_e = df_v[0].astype(str).str.lower().str.contains('e').any()
    print(f'  {len(archivos)} archivos | sin notación científica: {"✓" if not tiene_e else "✗"}')
    print(f'  Cortes: {cortes}')
    return archivos, cortes

def aplicar_oversampling(X_train, y_train, factor=2, pct=50, verbose=False):
    eps       = 1e-6
    vars_ok   = [v for v in VARS_PERFIL if v in X_train.columns]
    if not vars_ok:
        return X_train.copy(), y_train.copy()
    mask_b2   = y_train == 1
    X_b2      = X_train[mask_b2]
    umbrales  = {v: X_b2[v].quantile(pct / 100) for v in vars_ok}
    n_supera  = sum((X_b2[v] >= umbrales[v]).astype(int) for v in vars_ok)
    mask_fn   = n_supera >= min(2, len(vars_ok))
    X_fn      = X_b2[mask_fn]
    y_fn      = y_train[mask_b2][mask_fn]
    if len(X_fn) == 0:
        return X_train.copy(), y_train.copy()
    X_extra   = pd.concat([X_fn] * (factor - 1), ignore_index=True)
    y_extra   = pd.concat([y_fn] * (factor - 1), ignore_index=True)
    if verbose:
        print(f'    Oversampling: {len(X_fn):,} baja2 perfil "buen cliente" × {factor}')
    return (pd.concat([X_train, X_extra], ignore_index=True),
            pd.concat([y_train, y_extra], ignore_index=True))

print('✓ Funciones listas.')

## 2 · Carga de datos y params

In [ ]:
print('Cargando dataset_fe_v4.parquet...')
df = pd.read_parquet(CKPT_FE_V4)
df = df.sort_values([COL_ID, COL_PERIODO]).reset_index(drop=True)
print(f'Shape: {df.shape[0]:,} × {df.shape[1]} — {mem_mb(df):.1f} MB')

EXCLUIR      = {COL_ID, COL_PERIODO, TARGET_RAW, TARGET, 'es_baja'}
FEATURE_COLS = [c for c in df.columns if c not in EXCLUIR]
print(f'Features: {len(FEATURE_COLS):,}')

# Ratio de desbalance
mask_e        = df[COL_PERIODO].isin(MESES_CON_TARGET) & df[TARGET].notna()
y_ref         = df.loc[mask_e, TARGET]
SCALE_NATURAL = float((y_ref == 0).sum() / (y_ref == 1).sum())
del y_ref; gc.collect()
print(f'scale_pos_weight natural: {SCALE_NATURAL:.1f}')

# Params v5
with open(CKPT_V5) as f:
    p_v5 = json.load(f)

PARAMS_V5 = {k: v for k, v in p_v5.items()
             if k not in {'ganancia_optuna_fold4','n_trials','n_trials_completados',
                          'fecha','experimento','estado','n_meses_train','usar_is_unbalance'}}
PARAMS_V5.update({'objective':'binary','metric':'binary_logloss',
                  'verbosity':-1,'n_jobs':-1,'random_state':SEED})
if p_v5.get('usar_is_unbalance', False):
    PARAMS_V5['is_unbalance'] = True
    PARAMS_V5.pop('scale_pos_weight', None)
else:
    PARAMS_V5['scale_pos_weight'] = p_v5.get('scale_pos_weight', SCALE_NATURAL * 2)
    PARAMS_V5.pop('is_unbalance', None)

N_MESES_V5 = p_v5.get('n_meses_train', 3)
ES_DART    = PARAMS_V5.get('boosting_type') == 'dart'
G_V5_F4   = p_v5['ganancia_optuna_fold4']

print(f'Params v5: boosting={PARAMS_V5["boosting_type"]} | '
      f'ventana={N_MESES_V5} | ganancia_f4=${G_V5_F4:,.0f}')

FOLDS = [
    {'nombre':'fold_2','train':[202103],                'test':202104},
    {'nombre':'fold_3','train':[202103,202104],         'test':202105},
    {'nombre':'fold_4','train':[202103,202104,202105],  'test':202106},
]

def get_meses_train(fold_nombre, n_ventana):
    m = {'fold_2':[202103],'fold_3':[202103,202104],'fold_4':[202103,202104,202105]}
    return m[fold_nombre][-n_ventana:]

def get_split(meses_train, mes_test, feats=None):
    if feats is None: feats = FEATURE_COLS
    mt = df[COL_PERIODO].isin(meses_train) & df[TARGET].notna()
    me = (df[COL_PERIODO] == mes_test)      & df[TARGET].notna()
    return (df.loc[mt, feats].copy(), df.loc[mt, TARGET].astype(int).copy(),
            df.loc[me, feats].copy(), df.loc[me, TARGET].astype(int).copy())

---
## 3 · Experimento A — Oversampling manual

Corre los 3 configs manuales (×2 p50, ×3 p50, ×2 p75), elige el mejor en fold_4
y genera la submission. Sin Optuna — rápido.

In [ ]:
print('\n' + '='*60)
print('EXPERIMENTO A — OVERSAMPLING MANUAL (submissions7/)')
print('='*60)

configs_over = [
    {'nombre':'A1','factor':2,'pct':50},
    {'nombre':'A2','factor':3,'pct':50},
    {'nombre':'A3','factor':2,'pct':75},
]
resultados_over = {}

for cfg in configs_over:
    print(f"\n--- {cfg['nombre']}: ×{cfg['factor']} p{cfg['pct']} ---")
    for fold in FOLDS:
        meses_tr = get_meses_train(fold['nombre'], N_MESES_V5)
        Xtr, ytr, Xte, yte = get_split(meses_tr, fold['test'])
        Xtr_aug, ytr_aug = aplicar_oversampling(
            Xtr, ytr, factor=cfg['factor'], pct=cfg['pct'],
            verbose=(fold['nombre']=='fold_4'))
        modelo = lgb.LGBMClassifier(**PARAMS_V5)
        if ES_DART:
            modelo.fit(Xtr_aug, ytr_aug, callbacks=[lgb.log_evaluation(-1)])
        else:
            modelo.fit(Xtr_aug, ytr_aug,
                       eval_set=[(Xte, yte)],
                       callbacks=[lgb.early_stopping(50, verbose=False),
                                  lgb.log_evaluation(-1)])
        prob = modelo.predict_proba(Xte)[:, 1]
        res  = evaluar_fold(yte.values, prob, f"{cfg['nombre']} {fold['nombre']}")
        resultados_over.setdefault(cfg['nombre'], {})[fold['nombre']] = res
        del Xtr, ytr, Xtr_aug, ytr_aug, Xte, yte, modelo, prob; gc.collect()

# Elegir ganador
g_f4 = {k: v.get('fold_4',{}).get('ganancia',0) for k, v in resultados_over.items()}
ganador_a = max(g_f4, key=g_f4.get)
cfg_ganadora = next(c for c in configs_over if c['nombre'] == ganador_a)
print(f'\nGanancias fold_4:')
for k, g in g_f4.items():
    marca = ' ← GANADOR' if k == ganador_a else ''
    print(f'  {k}: ${g:,.0f} (vs v5: {"+ " if g>G_V5_F4 else ""}${abs(g-G_V5_F4)/1e6:.1f}M){marca}')

In [ ]:
# Walk-forward completo del ganador + semillerio
print(f'\nWalk-forward completo con {ganador_a}...')
rows_a = []
for fold in FOLDS:
    meses_tr = get_meses_train(fold['nombre'], N_MESES_V5)
    Xtr, ytr, Xte, yte = get_split(meses_tr, fold['test'])
    Xtr_aug, ytr_aug = aplicar_oversampling(
        Xtr, ytr, factor=cfg_ganadora['factor'], pct=cfg_ganadora['pct'])
    modelo = lgb.LGBMClassifier(**{**PARAMS_V5,'random_state':SEED})
    if ES_DART:
        modelo.fit(Xtr_aug, ytr_aug, callbacks=[lgb.log_evaluation(-1)])
    else:
        modelo.fit(Xtr_aug, ytr_aug,
                   eval_set=[(Xte,yte)],
                   callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(-1)])
    prob = modelo.predict_proba(Xte)[:,1]
    opt  = threshold_optimo(yte.values, prob)
    auc  = roc_auc_score(yte, prob)
    print(f'  {fold["nombre"]}: AUC={auc:.4f} | ${opt["ganancia"]:,.0f} | N={opt["n_intervenidos"]:,}')
    rows_a.append({'fold':fold['nombre'],'experimento':f'v11a_{ganador_a}',
                   'auc':round(auc,4),**opt})
    del Xtr,ytr,Xtr_aug,ytr_aug,Xte,yte,modelo,prob; gc.collect()

guardar_metricas(rows_a)
N_OPT_A = int(next(r for r in rows_a if r['fold']=='fold_4')['n_intervenidos'])
G_A_F4  = next(r for r in rows_a if r['fold']=='fold_4')['ganancia']

# Semillerio
TODOS = [202103,202104,202105,202106]
MESES_SUB = TODOS[-N_MESES_V5:]
mask_tf = df[COL_PERIODO].isin(MESES_SUB) & df[TARGET].notna()
X_tf    = df.loc[mask_tf, FEATURE_COLS].copy()
y_tf    = df.loc[mask_tf, TARGET].astype(int).copy()
mask_p  = df[COL_PERIODO] == MES_PREDECIR
X_pred  = df.loc[mask_p, FEATURE_COLS].copy()
ids_a   = df.loc[mask_p, COL_ID].values

X_tf_aug, y_tf_aug = aplicar_oversampling(X_tf, y_tf,
    factor=cfg_ganadora['factor'], pct=cfg_ganadora['pct'], verbose=True)
del X_tf, y_tf; gc.collect()

# Checkpoint probs
if CKPT_PROBS_A.exists() and CKPT_IDS_A.exists():
    probs_a = list(np.load(CKPT_PROBS_A))
    ids_ck  = np.load(CKPT_IDS_A)
    n_ya    = len(probs_a) if np.array_equal(ids_ck, ids_a) else 0
    if n_ya == 0: probs_a = []
    else: print(f'✓ Checkpoint A: {n_ya}/{len(SEMILLAS)} semillas')
else:
    probs_a = []; n_ya = 0

for i_s, sem in enumerate(SEMILLAS[n_ya:], start=n_ya):
    print(f'  A semilla {sem}...', end=' ')
    m = lgb.LGBMClassifier(**{**PARAMS_V5,'random_state':sem})
    m.fit(X_tf_aug, y_tf_aug, callbacks=[lgb.log_evaluation(-1)])
    probs_a.append(m.predict_proba(X_pred)[:,1])
    del m; gc.collect()
    np.save(CKPT_PROBS_A, np.array(probs_a))
    np.save(CKPT_IDS_A, ids_a)
    print('✓')

del X_tf_aug, y_tf_aug; gc.collect()
TS_A = datetime.now().strftime('%m%d_%H%M')
arch_a, cortes_a = generar_submission(probs_a, ids_a, N_OPT_A, RUTA_SUBS_A,
                                       f'v11a_{ganador_a}', TS_A)
print(f'\n✓ EXP A completo | fold_4: ${G_A_F4:,.0f} | '
      f'vs v5: {"+ " if G_A_F4>G_V5_F4 else ""}${abs(G_A_F4-G_V5_F4)/1e6:.1f}M')
print(f'submit v11a_{ganador_a}_{TS_A} | submissions7/')

---
## 4 · Experimento B — Kitchen Sink extendido

Retoma desde `CKPT_V5` como warm start y corre 50 trials adicionales.
Solo gbdt y goss (sin DART) para mantener tiempos razonables.

In [ ]:
print('\n' + '='*60)
print('EXPERIMENTO B — KITCHEN SINK EXTENDIDO (submissions8/)')
print('='*60)

if CKPT_V11B.exists():
    print('✓ CKPT_V11B encontrado — cargando...')
    with open(CKPT_V11B) as f:
        p_v11b = json.load(f)
    PARAMS_B = {k: v for k, v in p_v11b.items()
                if k not in {'ganancia_fold4','n_trials','fecha','experimento'}}
    PARAMS_B.update({'objective':'binary','metric':'binary_logloss',
                     'verbosity':-1,'n_jobs':-1,'random_state':SEED})
    G_B_F4   = p_v11b['ganancia_fold4']
    N_OPT_B  = p_v11b.get('n_opt', 10000)
    print(f'  Ganancia fold_4: ${G_B_F4:,.0f}')
    print('  → Saltá directo al semillerio B.')
else:
    print('Corriendo Optuna Kitchen Sink extendido...')

    KEYS_STRUCT = {'num_leaves','max_depth','learning_rate','n_estimators',
                   'min_child_samples','feature_fraction','bagging_fraction',
                   'bagging_freq','reg_alpha','reg_lambda','min_split_gain'}
    warm_b = {k: v for k, v in p_v5.items() if k in KEYS_STRUCT}

    # Pre-computa fold_4 split
    meses_b   = get_meses_train('fold_4', N_MESES_V5)
    Xtr_b, ytr_b, Xte_b, yte_b = get_split(meses_b, 202106)
    yte_b_arr = yte_b.values

    def objective_b(trial):
        boosting  = trial.suggest_categorical('boosting_type', ['gbdt','goss'])
        num_leaves= trial.suggest_int('num_leaves', 16, 256)
        max_depth = trial.suggest_int('max_depth', 4, 12)
        lr        = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        n_est     = trial.suggest_int('n_estimators', 100, 1000, step=50)
        min_child = trial.suggest_int('min_child_samples', 20, 2000, log=True)
        feat_frac = trial.suggest_float('feature_fraction', 0.4, 1.0)
        min_split = trial.suggest_float('min_split_gain', 0.0, 1.0)
        reg_a     = trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True)
        reg_l     = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True)

        if boosting == 'goss':
            bag_frac, bag_freq = 1.0, 0
        else:
            bag_frac = trial.suggest_float('bagging_fraction', 0.4, 1.0)
            bag_freq = trial.suggest_int('bagging_freq', 1, 7)

        usar_iu = trial.suggest_categorical('usar_is_unbalance', [True, False])
        params  = {'objective':'binary','metric':'binary_logloss','boosting_type':boosting,
                   'verbosity':-1,'n_jobs':-1,'random_state':SEED,
                   'num_leaves':num_leaves,'max_depth':max_depth,'learning_rate':lr,
                   'n_estimators':n_est,'min_child_samples':min_child,
                   'feature_fraction':feat_frac,'bagging_fraction':bag_frac,
                   'bagging_freq':bag_freq,'reg_alpha':reg_a,'reg_lambda':reg_l,
                   'min_split_gain':min_split}
        if usar_iu:
            params['is_unbalance'] = True
        else:
            params['scale_pos_weight'] = trial.suggest_float(
                'scale_pos_weight', SCALE_NATURAL/4, SCALE_NATURAL*5)

        modelo = lgb.LGBMClassifier(**params)
        modelo.fit(Xtr_b, ytr_b,
                   eval_set=[(Xte_b, yte_b)],
                   callbacks=[lgb.early_stopping(30,verbose=False),lgb.log_evaluation(-1)])
        prob     = modelo.predict_proba(Xte_b)[:,1]
        ganancia = threshold_optimo(yte_b_arr, prob)['ganancia']
        del modelo; gc.collect()
        return ganancia

    if 'study_b' not in globals():
        study_b = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=SEED),
            study_name='ks_v11b')
        warm_enq = {**warm_b,
            'boosting_type': p_v5.get('boosting_type','gbdt')
                             if p_v5.get('boosting_type') != 'dart' else 'gbdt',
            'usar_is_unbalance': p_v5.get('usar_is_unbalance', False),
            'min_split_gain': warm_b.get('min_split_gain', 0.0)}
        if not warm_enq['usar_is_unbalance']:
            warm_enq['scale_pos_weight'] = p_v5.get('scale_pos_weight', SCALE_NATURAL*2)
        if warm_enq['boosting_type'] != 'goss':
            warm_enq.setdefault('bagging_fraction',
                                warm_b.get('bagging_fraction', 0.8))
            warm_enq.setdefault('bagging_freq', warm_b.get('bagging_freq', 1))
        study_b.enqueue_trial(warm_enq)
        print('Study B creado con warm start desde v5.')
    else:
        ok = len([t for t in study_b.trials if t.value is not None])
        print(f'Study B existente — {ok} trials, retomando...')

    restantes = N_TRIALS_B - len([t for t in study_b.trials if t.value is not None])
    print(f'Corriendo {restantes} trials...')
    study_b.optimize(objective_b, n_trials=restantes, show_progress_bar=True)
    del Xtr_b, ytr_b, Xte_b; gc.collect()

    bp = study_b.best_params
    PARAMS_B = {'objective':'binary','metric':'binary_logloss',
                'boosting_type':bp['boosting_type'],'verbosity':-1,
                'n_jobs':-1,'random_state':SEED,
                'num_leaves':bp['num_leaves'],'max_depth':bp['max_depth'],
                'learning_rate':bp['learning_rate'],'n_estimators':bp['n_estimators'],
                'min_child_samples':bp['min_child_samples'],
                'feature_fraction':bp['feature_fraction'],
                'bagging_fraction':bp.get('bagging_fraction',1.0),
                'bagging_freq':bp.get('bagging_freq',0),
                'reg_alpha':bp['reg_alpha'],'reg_lambda':bp['reg_lambda'],
                'min_split_gain':bp['min_split_gain']}
    if bp.get('usar_is_unbalance'):
        PARAMS_B['is_unbalance'] = True
    else:
        PARAMS_B['scale_pos_weight'] = bp.get('scale_pos_weight', SCALE_NATURAL*2)

    # Walk-forward para obtener N_OPT_B y G_B_F4
    print('\nWalk-forward B...')
    rows_b = []
    for fold in FOLDS:
        mtr = get_meses_train(fold['nombre'], N_MESES_V5)
        Xtr,ytr,Xte,yte = get_split(mtr, fold['test'])
        m = lgb.LGBMClassifier(**{**PARAMS_B,'random_state':SEED})
        m.fit(Xtr,ytr,eval_set=[(Xte,yte)],
              callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(-1)])
        prob = m.predict_proba(Xte)[:,1]
        opt  = threshold_optimo(yte.values,prob)
        auc  = roc_auc_score(yte,prob)
        print(f'  {fold["nombre"]}: ${opt["ganancia"]:,.0f} | N={opt["n_intervenidos"]:,}')
        rows_b.append({'fold':fold['nombre'],'experimento':'v11b_ks_ext',
                       'auc':round(auc,4),**opt})
        del Xtr,ytr,Xte,yte,m,prob; gc.collect()

    guardar_metricas(rows_b)
    G_B_F4  = next(r for r in rows_b if r['fold']=='fold_4')['ganancia']
    N_OPT_B = int(next(r for r in rows_b if r['fold']=='fold_4')['n_intervenidos'])

    with open(CKPT_V11B,'w') as f:
        json.dump({**bp,'ganancia_fold4':G_B_F4,'n_opt':N_OPT_B,
                   'n_trials':N_TRIALS_B,'experimento':'v11b_ks_ext',
                   'fecha':datetime.now().isoformat()}, f, indent=2)
    print(f'✓ CKPT_V11B guardado | fold_4: ${G_B_F4:,.0f}')

In [ ]:
# Semillerio B
print(f'\nSemillerio B | N_OPT={N_OPT_B:,}')
MESES_SUB_B = [202103,202104,202105,202106][-N_MESES_V5:]
mask_tf = df[COL_PERIODO].isin(MESES_SUB_B) & df[TARGET].notna()
X_tf_b  = df.loc[mask_tf, FEATURE_COLS].copy()
y_tf_b  = df.loc[mask_tf, TARGET].astype(int).copy()
mask_p  = df[COL_PERIODO] == MES_PREDECIR
X_pred_b= df.loc[mask_p, FEATURE_COLS].copy()
ids_b   = df.loc[mask_p, COL_ID].values

if CKPT_PROBS_B.exists() and CKPT_IDS_B.exists():
    probs_b = list(np.load(CKPT_PROBS_B))
    ids_ck  = np.load(CKPT_IDS_B)
    n_ya    = len(probs_b) if np.array_equal(ids_ck, ids_b) else 0
    if n_ya == 0: probs_b = []
    else: print(f'✓ Checkpoint B: {n_ya}/{len(SEMILLAS)} semillas')
else:
    probs_b = []; n_ya = 0

for i_s, sem in enumerate(SEMILLAS[n_ya:], start=n_ya):
    print(f'  B semilla {sem}...', end=' ')
    m = lgb.LGBMClassifier(**{**PARAMS_B,'random_state':sem})
    m.fit(X_tf_b, y_tf_b, callbacks=[lgb.log_evaluation(-1)])
    probs_b.append(m.predict_proba(X_pred_b)[:,1])
    del m; gc.collect()
    np.save(CKPT_PROBS_B, np.array(probs_b))
    np.save(CKPT_IDS_B, ids_b)
    print('✓')

del X_tf_b, y_tf_b, X_pred_b; gc.collect()
TS_B = datetime.now().strftime('%m%d_%H%M')
arch_b, cortes_b = generar_submission(probs_b, ids_b, N_OPT_B,
                                       RUTA_SUBS_B, 'v11b_ks', TS_B)
print(f'\n✓ EXP B completo | fold_4: ${G_B_F4:,.0f} | '
      f'vs v5: {"+ " if G_B_F4>G_V5_F4 else ""}${abs(G_B_F4-G_V5_F4)/1e6:.1f}M')
print(f'submit v11b_ks_{TS_B} | submissions8/')

---
## 5 · Experimento C — Train con 4 meses completos

Mismos params del v5, pero el modelo final entrena con **todos** los meses
disponibles con target (202103+04+05+06) en vez de los últimos 3.
El walk-forward sigue igual — solo cambia el train del modelo final.

In [ ]:
print('\n' + '='*60)
print('EXPERIMENTO C — 4 MESES COMPLETOS (submissions9/)')
print('='*60)

# Walk-forward igual que v5 para comparar
print('Walk-forward C (igual que v5 para referencia)...')
rows_c = []
for fold in FOLDS:
    mtr = get_meses_train(fold['nombre'], N_MESES_V5)
    Xtr,ytr,Xte,yte = get_split(mtr, fold['test'])
    m = lgb.LGBMClassifier(**{**PARAMS_V5,'random_state':SEED})
    if ES_DART:
        m.fit(Xtr,ytr,callbacks=[lgb.log_evaluation(-1)])
    else:
        m.fit(Xtr,ytr,eval_set=[(Xte,yte)],
              callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(-1)])
    prob = m.predict_proba(Xte)[:,1]
    opt  = threshold_optimo(yte.values,prob)
    auc  = roc_auc_score(yte,prob)
    print(f'  {fold["nombre"]}: ${opt["ganancia"]:,.0f} | N={opt["n_intervenidos"]:,}')
    rows_c.append({'fold':fold['nombre'],'experimento':'v11c_4meses',
                   'auc':round(auc,4),**opt})
    del Xtr,ytr,Xte,yte,m,prob; gc.collect()

guardar_metricas(rows_c)
G_C_F4  = next(r for r in rows_c if r['fold']=='fold_4')['ganancia']
N_OPT_C = int(next(r for r in rows_c if r['fold']=='fold_4')['n_intervenidos'])
print(f'\nWalk-forward C fold_4: ${G_C_F4:,.0f}')

# Semillerio con TODOS los 4 meses en train final
print('\nSemillerio C — train con 4 meses completos...')
mask_tf = df[COL_PERIODO].isin([202103,202104,202105,202106]) & df[TARGET].notna()
X_tf_c  = df.loc[mask_tf, FEATURE_COLS].copy()
y_tf_c  = df.loc[mask_tf, TARGET].astype(int).copy()
mask_p  = df[COL_PERIODO] == MES_PREDECIR
X_pred_c= df.loc[mask_p, FEATURE_COLS].copy()
ids_c   = df.loc[mask_p, COL_ID].values
print(f'  Train: {len(y_tf_c):,} (baja2={y_tf_c.sum():,}) | 4 meses completos')

if CKPT_PROBS_C.exists() and CKPT_IDS_C.exists():
    probs_c = list(np.load(CKPT_PROBS_C))
    ids_ck  = np.load(CKPT_IDS_C)
    n_ya    = len(probs_c) if np.array_equal(ids_ck, ids_c) else 0
    if n_ya == 0: probs_c = []
    else: print(f'✓ Checkpoint C: {n_ya}/{len(SEMILLAS)} semillas')
else:
    probs_c = []; n_ya = 0

for i_s, sem in enumerate(SEMILLAS[n_ya:], start=n_ya):
    print(f'  C semilla {sem}...', end=' ')
    m = lgb.LGBMClassifier(**{**PARAMS_V5,'random_state':sem})
    if ES_DART:
        m.fit(X_tf_c, y_tf_c, callbacks=[lgb.log_evaluation(-1)])
    else:
        m.fit(X_tf_c, y_tf_c, callbacks=[lgb.log_evaluation(-1)])
    probs_c.append(m.predict_proba(X_pred_c)[:,1])
    del m; gc.collect()
    np.save(CKPT_PROBS_C, np.array(probs_c))
    np.save(CKPT_IDS_C, ids_c)
    print('✓')

del X_tf_c, y_tf_c, X_pred_c; gc.collect()
TS_C = datetime.now().strftime('%m%d_%H%M')
arch_c, cortes_c = generar_submission(probs_c, ids_c, N_OPT_C,
                                       RUTA_SUBS_C, 'v11c_4m', TS_C)
print(f'\n✓ EXP C completo | fold_4: ${G_C_F4:,.0f} | '
      f'vs v5: {"+ " if G_C_F4>G_V5_F4 else ""}${abs(G_C_F4-G_V5_F4)/1e6:.1f}M')
print(f'submit v11c_4m_{TS_C} | submissions9/')

## 6 · Resumen final

In [ ]:
del df; gc.collect()

print('\n' + '='*60)
print('RESUMEN FINAL — Notebook 11')
print('='*60)
print(f'\n{"Experimento":>25} | {"Ganancia fold_4":>18} | {"vs v5":>12} | {"Submission"}')
print('─'*75)
for nombre, g, sub in [
    ('v5 base (referencia)',   G_V5_F4, '—'),
    (f'A: oversampling {ganador_a}', G_A_F4, f'submissions7/ ({TS_A})'),
    ('B: kitchen sink ext',   G_B_F4, f'submissions8/ ({TS_B})'),
    ('C: 4 meses completos',  G_C_F4, f'submissions9/ ({TS_C})'),
]:
    delta = g - G_V5_F4
    signo = '+' if delta > 0 else ''
    marca = ' ✓' if g > G_V5_F4 else ''
    print(f'{nombre:>25} | ${g:>15,.0f} | {signo}${delta/1e6:>8.1f}M{marca} | {sub}')

mejor = max([('A',G_A_F4),('B',G_B_F4),('C',G_C_F4)], key=lambda x: x[1])
print(f'\n→ Mejor experimento: {mejor[0]} (${mejor[1]:,.0f})')
print('\nPrioridad de submits al despertar:')
ranking = sorted([('A',G_A_F4,'submissions7'),
                  ('B',G_B_F4,'submissions8'),
                  ('C',G_C_F4,'submissions9')],
                 key=lambda x: -x[1])
for i,(exp,g,sub) in enumerate(ranking, 1):
    print(f'  {i}. Exp {exp}: ${g:,.0f} → {sub}/')
print('\n✓ Notebook 11 completada.')